**Eye-Tracking & Neurocognitive Intelligence | Eye-Movement Modelling & Advanced AI | Presented by: Reza Saadatyar (2026)**

**03 | CNN and TCN on the Frozen 100-Person Cohort (pending execution)**

![Independent temporal model architectures and validation within source-participant partitions](../../images/03_cnn_and_tcn_models.svg)

Notebook 01 has prepared and committed **100 participants, 400 recordings and 9,600 normalised windows**, and Notebook 02 has completed main-cohort classical Validation. This notebook fits **CNN and TCN** on the frozen **64/16/20** participant Train/Validation/reserved Test split. The currently tracked `03_*` numerical outputs are still from the ten-person pilot until this run replaces them.

A fresh Colab checkout already contains the **current stage-01 temporal archive and split files**; no raw-data download or ZIP upload is needed. If explicitly uploading a matching `eye_movement_modelling_Notebook_01_Outputs.zip` under `/content/`, that archive takes precedence. Never reuse an older pilot ZIP. The notebook checks participant and recording identities before training, and does not access the reserved Test for scoring.


**00. Colab Repository Setup**


In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**


In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("eye_movement_modelling")

# -------------------- 2. Temporal Training and Evaluation --------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from IPython.display import display
from s2_handoff import ensure_s2_stage_inputs, verify_main_cohort
from core.output_download import download_outputs_from_colab
from core.plot_style import plot_style, save_figure

# ------------------------- 3. Study Model Utilities --------------------------
from s2_data import (
    TASKS, load_protocol, load_windows, verify_temporal_source_alignment,
)
from s2_training import fit_temporal
from s2_metrics import metric_row

# ----------------------------- 4. Output Folder ------------------------------
output_dir = root / "eye_movement_modelling" / "output"
output_dir.mkdir(parents=True, exist_ok=True)
plot_style(font_family="DejaVu Sans", figure_dpi=100)
print("Eye-Movement Modelling temporal outputs:", output_dir)


**02. Current Temporal Input Contract**

In a fresh Runtime, read committed stage-01 files; a matching uploaded Notebook 01 ZIP under `/content/` is optional. Validate source records, four channels, temporal dimensions and the participant-disjoint main split. The main cohort has **9,600 windows**: **6,144 Train**, **1,536 Validation** and **1,920 reserved Test**. The frozen record keys, labels and channel dimensions are verified before fitting. This section is pending execution on the chosen main cohort. Linked figures remain historical pilot illustrations until the new run produces updated numerical figures.


In [ ]:
# ==================== 02. Recover Frozen Temporal Inputs =====================

# -------------------- 1. Restore Explicit Stage 01 Files ---------------------
ensure_s2_stage_inputs(output_dir, 1, (
    "01_sequence_windows.npz",
    "01_split_assignments.csv", "01_protocol.csv",
))
windows = load_windows(output_dir / "01_sequence_windows.npz")
split = pd.read_csv(
    output_dir / "01_split_assignments.csv",
    dtype={"record_id": str, "participant_id": str},
)
protocol = load_protocol(output_dir / "01_protocol.csv")
role_counts = verify_main_cohort(protocol, split)
print("Main cohort participant roles:", role_counts)
if windows["x"].shape[1:] != (
    protocol["window_samples"], len(protocol["temporal_channels"]),
):
    raise ValueError("Sequence data does not match the saved source protocol")

# -------------------------- 2. Display Data Support --------------------------
record_roles = dict(zip(split.record_id, split.split))
alignment_counts = verify_temporal_source_alignment(windows, split)
roles = np.array([record_roles[str(r)] for r in windows["record_id"]])
print("Verified window counts by participant partition:", alignment_counts)
validation_people = split.loc[
    split.split.eq("validation"), "participant_id",
].nunique()
print("Study mode:", protocol["purpose"])
print("Sequence tensor:", windows["x"].shape)
print("Train/Validation/Test window counts:")
display(pd.Series(roles).value_counts().rename("n_windows").to_frame())
print("Independent Validation participants:", validation_people)
reported = []


**03. Train 1D-CNN**

Fit the existing CNN on new Train windows only. Select checkpoints using record-level Validation Macro F1, with window loss as the tiebreaker. Save current metrics and curves. This section is pending execution on the chosen main cohort. Linked figures remain historical pilot illustrations until the new run produces updated numerical figures.


In [ ]:
# ============================= 03. Train 1D-CNN ==============================

# --------------------- 1. Fit on Train Participants Only ---------------------
model_name = "cnn"
history, predictions, info = fit_temporal(
    model_name, windows, split, output_dir / f"03_cnn.pt",
    seed=protocol["split_seed"], epochs=8, patience=3,
    batch_size=64,
)
history.to_csv(output_dir / f"03_cnn_history.csv", index=False)
predictions.to_csv(
    output_dir / f"03_cnn_validation_predictions.csv", index=False,
)
print("Model:", model_name)
print("Best epoch:", info["best_epoch"])
print("Trainable parameters:", info["parameters"])
display(history.tail(8))

# --------------------- 2. Report Record-Level Validation ---------------------
result = metric_row(
    predictions["target_index"], predictions["pred_index"],
    model=model_name, partition="validation",
    n_participants=validation_people,
)
result["parameters"] = int(info["parameters"])
result["best_epoch"] = int(info["best_epoch"])
reported.append(result)
display(pd.DataFrame([result]))

# ------------------ 3. Display Learning and Error Evidence -------------------
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))
axes[0].plot(history["epoch"], history["train_loss"],
             marker="o", label="Train loss")
axes[0].plot(history["epoch"], history["validation_window_loss"],
             marker="o", label="Validation loss")
axes[0].set(xlabel="Epoch", ylabel="Cross-entropy",
            title="1D-CNN: Learning Curves")
axes[0].grid(alpha=0.2)
axes[0].legend()
matrix = confusion_matrix(
    predictions.target_index, predictions.pred_index,
    labels=range(len(TASKS)),
)
ConfusionMatrixDisplay(matrix, display_labels=TASKS).plot(
    ax=axes[1], colorbar=False, values_format="d",
)
axes[1].set_title("1D-CNN: Validation by Record")
fig.tight_layout()
save_figure(fig, output_dir / f"03_cnn_evaluation.png")
plt.show()


**04. Train TCN**

Fit the existing TCN with matching people and temporal representation. Save the Validation-selected checkpoint, curves and per-record predictions. This section is pending execution on the chosen main cohort. Linked figures remain historical pilot illustrations until the new run produces updated numerical figures.


In [ ]:
# ============== 04. Train Temporal Convolutional Network (TCN) ===============

# --------------------- 1. Fit on Train Participants Only ---------------------
model_name = "tcn"
history, predictions, info = fit_temporal(
    model_name, windows, split, output_dir / f"03_tcn.pt",
    seed=protocol["split_seed"], epochs=8, patience=3,
    batch_size=64,
)
history.to_csv(output_dir / f"03_tcn_history.csv", index=False)
predictions.to_csv(
    output_dir / f"03_tcn_validation_predictions.csv", index=False,
)
print("Model:", model_name)
print("Best epoch:", info["best_epoch"])
print("Trainable parameters:", info["parameters"])
display(history.tail(8))

# --------------------- 2. Report Record-Level Validation ---------------------
result = metric_row(
    predictions["target_index"], predictions["pred_index"],
    model=model_name, partition="validation",
    n_participants=validation_people,
)
result["parameters"] = int(info["parameters"])
result["best_epoch"] = int(info["best_epoch"])
reported.append(result)
display(pd.DataFrame([result]))

# ------------------ 3. Display Learning and Error Evidence -------------------
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))
axes[0].plot(history["epoch"], history["train_loss"],
             marker="o", label="Train loss")
axes[0].plot(history["epoch"], history["validation_window_loss"],
             marker="o", label="Validation loss")
axes[0].set(xlabel="Epoch", ylabel="Cross-entropy",
            title="Temporal Convolutional Network (TCN): Learning Curves")
axes[0].grid(alpha=0.2)
axes[0].legend()
matrix = confusion_matrix(
    predictions.target_index, predictions.pred_index,
    labels=range(len(TASKS)),
)
ConfusionMatrixDisplay(matrix, display_labels=TASKS).plot(
    ax=axes[1], colorbar=False, values_format="d",
)
axes[1].set_title("Temporal Convolutional Network (TCN): Validation by Record")
fig.tight_layout()
save_figure(fig, output_dir / f"03_tcn_evaluation.png")
plt.show()


**05. Current-Cohort Model Comparison**

Compare CNN and TCN on the same Validation recordings. Save model parameters, best epochs, Macro F1, Balanced Accuracy and accuracy without opening Test. This section is pending execution on the chosen main cohort. Linked figures remain historical pilot illustrations until the new run produces updated numerical figures.


In [ ]:
# ====================== 05. Compare Temporal Validation ======================

# --------------------- 1. Save Shared Validation Metrics ---------------------
metrics = pd.DataFrame(reported).sort_values(
    ["macro_f1", "balanced_accuracy", "model"],
    ascending=[False, False, True],
)
metrics.to_csv(
    output_dir / "03_neural_validation_metrics.csv", index=False,
)
display(metrics)

# --------------------- 2. Plot Independent Model Scores ----------------------
fig, ax = plt.subplots(figsize=(7.8, 4.5))
pos = np.arange(len(metrics))
ax.bar(pos - 0.16, metrics["macro_f1"], 0.32, label="Macro F1")
ax.bar(pos + 0.16, metrics["balanced_accuracy"], 0.32,
       label="Balanced accuracy")
ax.set_xticks(pos, metrics["model"])
ax.set(ylabel="Record-level Validation score", ylim=(0, 1),
       title="1D-CNN and Temporal Convolutional Network (TCN): Validation Comparison")
ax.legend()
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
save_figure(fig, output_dir / "03_model_comparison.png")
plt.show()


**06. Interpretation and Stage ZIP**

Before exporting, confirm both current model prediction tables cover the exact original Validation records with matching task labels, probabilities and metrics. Then package current checkpoints, histories and Validation evidence into a stage-03 ZIP for Notebook 05. Keep any file above 50,000,000 bytes out of Git. This section is pending execution on the chosen main cohort. Linked figures remain historical pilot illustrations until the new run produces updated numerical figures.


In [ ]:
# ======================= 06. Export Stage-03 Evidence ========================

# -------------------------- 1. Confirm Stage Files ---------------------------
stage03_files = [
    "03_neural_validation_metrics.csv",
    "03_model_comparison.png",
]
for model_name in ("cnn", "tcn"):
    stage03_files.extend((
        f"03_{model_name}.pt",
        f"03_{model_name}_history.csv",
        f"03_{model_name}_validation_predictions.csv",
        f"03_{model_name}_evaluation.png",
    ))
missing = [name for name in stage03_files
           if not (output_dir / name).is_file()]
if missing:
    raise FileNotFoundError(f"Incomplete stage-03 outputs: {missing}")

# ---------------------- 2. Check Source and Score Alignment ------------------
expected = split.loc[
    split.split.eq("validation"),
    ["record_id", "participant_id", "task"],
].set_index("record_id")
if expected.index.has_duplicates:
    raise ValueError("Duplicate Validation source records")
expected_ids = set(expected.index)
expected_y = expected["task"].map(
    {task: index for index, task in enumerate(TASKS)}
).sort_index()
window_counts = pd.Series(
    windows["record_id"].astype(str)
).value_counts()
summary = pd.read_csv(output_dir / "03_neural_validation_metrics.csv")
if len(summary) != 2 or set(summary.model) != {"cnn", "tcn"}:
    raise ValueError("Main Validation metrics require CNN and TCN")
probability_columns = [f"p_{label}" for label in TASKS]

for model_name in ("cnn", "tcn"):
    saved = pd.read_csv(
        output_dir / f"03_{model_name}_validation_predictions.csv",
        dtype={"record_id": str},
    )
    if (
        saved.record_id.duplicated().any()
        or set(saved.record_id) != expected_ids
        or not saved.model.eq(model_name).all()
        or not saved.partition.eq("validation").all()
    ):
        raise ValueError(
            f"{model_name} predictions do not match main Validation records"
        )
    observed = saved.set_index("record_id").sort_index()
    if not observed.target_index.eq(expected_y).all():
        raise ValueError(f"{model_name} source-task labels differ")
    expected_n = window_counts.reindex(observed.index)
    if (
        expected_n.isna().any()
        or not observed.n_windows.eq(expected_n).all()
    ):
        raise ValueError(f"{model_name} window counts differ from source")
    probabilities = observed[probability_columns].to_numpy(dtype=float)
    if (
        not np.isfinite(probabilities).all()
        or not np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-4)
        or not np.array_equal(
            probabilities.argmax(axis=1),
            observed.pred_index.to_numpy(dtype=int),
        )
    ):
        raise ValueError(f"{model_name} probabilities or labels are invalid")
    actual = metric_row(
        observed.target_index, observed.pred_index,
        model=model_name, partition="validation",
        n_participants=expected.participant_id.nunique(),
    )
    recorded = summary.loc[summary.model.eq(model_name)].iloc[0]
    for key in ("accuracy", "macro_f1", "balanced_accuracy"):
        if not np.isclose(actual[key], float(recorded[key]), atol=1e-9):
            raise ValueError(f"{model_name} {key} disagrees with predictions")
    if (
        int(recorded.n_records) != len(expected)
        or int(recorded.n_participants) != expected.participant_id.nunique()
    ):
        raise ValueError(f"{model_name} reported support differs")

print(
    "Validated both models against",
    len(expected_ids), "main-cohort Validation recordings.",
)

# ------------------------ 3. Export Stage-Owned Outputs ----------------------
download_outputs_from_colab("eye_movement_modelling/output", "03")
